# 第 6 章：类方法、静态方法与抽象基类

> 本章目标：分清**实例方法 / 类方法 / 静态方法**三种方法的适用边界，掌握 `@classmethod` 工厂模式与 `from_XXX` 惯用法，学会用 `abc` 模块设计强制性接口。

---

## 6.1 三种方法一张图

```mermaid
flowchart TD
    M[方法] --> I[实例方法<br/>def m self]
    M --> C[类方法<br/>@classmethod<br/>def m cls]
    M --> S[静态方法<br/>@staticmethod<br/>def m ...]
    I --> I1[第一个参数 self = 实例<br/>能读写实例状态]
    C --> C1[第一个参数 cls = 类<br/>能访问类属性/创建实例]
    S --> S1[没有自动参数<br/>就是放在类里的普通函数]
```

| 对比 | 实例方法 | 类方法 | 静态方法 |
|------|---------|--------|---------|
| 装饰器 | 无 | `@classmethod` | `@staticmethod` |
| 隐式首参 | `self`（实例） | `cls`（类） | 无 |
| 访问实例属性 | ✅ | ❌（拿到的是类） | ❌ |
| 访问类属性 | ✅（经 `type(self)` 或类名） | ✅ | ❌（需硬编码类名） |
| 能否被子类正确感知 | ✅ | ✅ `cls` 自动是实际调用的类 | ❌ |
| 典型用途 | 操作具体对象 | 工厂方法、操作类级状态 | 工具函数（与类相关的纯函数） |

In [1]:
class Pizza:
    default_size = 9            # 类属性

    def __init__(self, size, toppings):
        self.size = size                # 实例属性
        self.toppings = toppings

    # 1) 实例方法：操作具体某张披萨
    def describe(self):
        return f"{self.size} 寸披萨，配料: {self.toppings}"

    # 2) 类方法：操作类级别的信息
    @classmethod
    def default(cls):
        """工厂方法：用默认配置造一张披萨"""
        return cls(cls.default_size, ["芝士"])

    # 3) 静态方法：与披萨相关的工具函数，不需要 self/cls
    @staticmethod
    def calc_price(size, toppings):
        """纯计算：尺寸底价 + 每份配料 5 元"""
        return size * 2 + len(toppings) * 5


p = Pizza(12, ["芝士", "菠萝"])
print(p.describe())                 # 实例方法
print(Pizza.default().describe())   # 类方法当工厂
print(Pizza.calc_price(12, ["芝士", "菠萝"]))  # 静态方法

12 寸披萨，配料: ['芝士', '菠萝']
9 寸披萨，配料: ['芝士']
34


## 6.2 `@classmethod` 的核心价值一：工厂方法

一个类常常需要**多种创建方式**。`__init__` 只有一个，剩下的用 `from_XXX` 类方法解决：

> 💡 **对比 Java**：Java 用重载多个构造函数实现；Python 的 `__init__` 不支持重载，`from_XXX` 类方法是标准替代品。

```mermaid
flowchart LR
    S1[字符串 '2024-06-01'] -->|Date.from_string| D[Date 对象]
    S2[时间戳 1717200000] -->|Date.from_timestamp| D
    S3[年月日三参数] -->|Date 年 月 日| D
```

In [2]:
class Date:
    def __init__(self, year, month, day):
        self.year, self.month, self.day = year, month, day

    def __repr__(self):
        return f"Date({self.year}-{self.month:02d}-{self.day:02d})"

    @classmethod
    def from_string(cls, s):
        """从 'YYYY-MM-DD' 字符串创建"""
        year, month, day = map(int, s.split("-"))
        return cls(year, month, day)     # 注意用 cls 而不是硬编码 Date

    @classmethod
    def today(cls):
        """用系统当前日期创建"""
        import datetime
        t = datetime.date.today()
        return cls(t.year, t.month, t.day)


print(Date(2024, 6, 1))
print(Date.from_string("2024-06-01"))
print(Date.today())

# cls 的妙处：子类调用工厂方法，造出来的是子类！
class ChineseDate(Date):
    def __repr__(self):
        return f"{self.year}年{self.month}月{self.day}日"


print(ChineseDate.from_string("2024-06-01"))   # 2024年6月1日，不是 Date

Date(2024-06-01)
Date(2024-06-01)
Date(2026-08-23)
2024年6月1日


## 6.3 `@classmethod` 的核心价值二：继承感知

看这个反例--**在静态方法里硬编码类名**，会悄悄破坏继承：

| 场景 | `@classmethod` + `cls` | 硬编码 `Pizza.xxx` / 静态方法 |
|------|----------------------|-------------------------------|
`Pizza.default()` | `cls` 是 `Pizza` ✅ | 返回 `Pizza` ✅ |
`MyPizza.default()`（子类调用） | `cls` 自动变成 `MyPizza` ✅ | 仍然返回 `Pizza` ❌ |

结论：**只要方法内需要"当前的类"（创建实例、访问类属性），就用 `@classmethod`**。

In [3]:
class Base:
    count = 0

    @classmethod
    def register(cls):
        cls.count += 1          # cls 是实际调用者 -> 子类有自己的 count


class Child(Base):
    pass


Child.register()
Child.register()
print(f"Base.count = {Base.count}")    # 0 -- 没被"污染"
print(f"Child.count = {Child.count}")  # 2 -- 落在子类的命名空间
# 若 register 里写的是 Base.count，两个结果都会是 2 -- 这就是硬编码的坑

Base.count = 0
Child.count = 2


## 6.4 `@staticmethod`：放对位置的普通函数

静态方法就是**被收纳进类命名空间的普通函数**。它不接收 `self`/`cls`，存在的意义是**表达归属**："这个函数和这个类关系密切"。

✅ 适合：纯计算、格式转换、校验函数，既不读实例状态也不读类状态。
❌ 滥用：如果函数和类没什么关系，直接写成模块级函数更 Pythonic。

> 💡 什么时候"升级"成静态方法？当它是**私有工具**、只被这个类的方法调用时--放进类里可以减少模块命名空间的噪音。

In [4]:
class EmailValidator:
    @staticmethod
    def is_valid(email):
        """与邮箱相关的纯校验函数：不碰实例、不碰类"""
        return "@" in email and "." in email.split("@")[-1]

    @classmethod
    def assert_valid(cls, email):
        if not cls.is_valid(email):    # 类内部互调用
            raise ValueError(f"非法邮箱: {email}")
        print(f"{email} 校验通过")


print(EmailValidator.is_valid("a@b.com"))
EmailValidator.assert_valid("user@example.com")
try:
    EmailValidator.assert_valid("bad-email")
except ValueError as e:
    print(e)

True
user@example.com 校验通过
非法邮箱: bad-email


## 6.5 抽象基类（ABC）：强制契约

第 4 章已经见过 ABC 的多态用法，这里系统地过一遍。

**抽象基类（Abstract Base Class）** = 声明了"必须有但我不实现"的方法的类。

| 组件 | 作用 |
|------|------|
| `abc.ABC` | 继承它，你的类就成了抽象基类 |
| `@abstractmethod` | 标记"子类必须实现"的方法 |
| `@abstractmethod` + `@property` | 抽象属性（第 2 章 property 的抽象版） |
| `abc.register` | 把无关类"虚拟注册"为子类（高级用法，isinstance 认但无继承） |

```mermaid
flowchart TD
    A[定义 Shape ABC] --> B[abstractmethod: area]
    B --> C[Circle 实现了 area]
    C --> D[✅ 可以实例化]
    B --> E[Triangle 没实现 area]
    E --> F[❌ 实例化时 TypeError]
    style D fill:#d4f7d4
    style F fill:#ffd4d4
```

> 💡 **对比 Java**：ABC ≈ 抽象类 + 抽象方法；纯"接口"效果则对应 Java 的 `interface`（Python 没有独立 interface 关键字，ABC 或 Protocol 承担了这个角色）。
> **对比 C++**：ABC ≈ 含纯虚函数 `virtual ... = 0` 的类。

In [5]:
from abc import ABC, abstractmethod

class Storage(ABC):
    """存储后端的统一接口"""

    @abstractmethod
    def save(self, key, value):
        """子类必须实现"""

    @abstractmethod
    def load(self, key):
        """子类必须实现"""

    def dump(self):
        """具体方法：复用抽象接口写公共逻辑（模板方法模式）"""
        return f"<{type(self).__name__}> 就绪"


class MemoryStorage(Storage):
    def __init__(self):
        self.data = {}

    def save(self, key, value):
        self.data[key] = value

    def load(self, key):
        return self.data.get(key)


class BrokenStorage(Storage):
    """漏了 load 方法"""
    def save(self, key, value):
        pass


mem = MemoryStorage()
mem.save("user", "张三")
print(mem.load("user"))
print(mem.dump())

try:
    BrokenStorage()    # 缺抽象方法 -> 实例化即报错，错误在"入口"被发现
except TypeError as e:
    print(f"契约检查: {e}")

张三
<MemoryStorage> 就绪
契约检查: Can't instantiate abstract class BrokenStorage without an implementation for abstract method 'load'


## 6.6 抽象方法也可以有默认实现

抽象方法**可以有方法体**：子类 `super().xxx()` 能调用它，实现"每个子类必须处理，但有个兜底逻辑"的模板模式：

In [6]:
from abc import ABC, abstractmethod

class Report(ABC):
    @abstractmethod
    def header(self):
        """有默认实现的抽象方法：即使有方法体，子类也必须覆盖"""
        return "=== 报告 ==="

    @abstractmethod
    def body(self):
        ...   # 无默认实现，纯契约

    def render(self):
        """模板方法：流程固定，细节交给子类"""
        print(self.header())
        print(self.body())


class SalesReport(Report):
    def header(self):
        return "【销售月报】"          # 覆盖默认，写自己的

    def body(self):
        return "本月销售额 100 万"

class PlainReport(Report):
    def header(self):
        return super().header()       # 覆盖但复用父类兜底逻辑

    def body(self):
        return "一切正常"


SalesReport().render()
PlainReport().render()

【销售月报】
本月销售额 100 万
=== 报告 ===
一切正常


## 6.7 常见误区盘点

1. **静态方法里用 `self`** -- 编译能过（就是个普通函数），但说明它本该是实例方法；
2. **类方法里创建实例时硬编码类名** -- `return Pizza(...)` 而不是 `return cls(...)`，子类工厂悄悄失效；
3. **把 ABC 当普通父类继承却不实现抽象方法** -- 以为实现了，写错方法名照样报错（好在这正是 ABC 的价值）；
4. **用 ABC 检查鸭子类型** -- `isinstance(x, MyABC)` 只对注册/继承关系有效，纯鸭子类型请用 `Protocol`（第 4 章）。

## 6.8 本章小结

一句话选型：

```mermaid
flowchart TD
    Q{这个方法需要什么?}
    Q -- 实例的数据 --> A[实例方法]
    Q -- 类本身/要创建实例 --> B[classmethod]
    Q -- 什么都不需要，只是逻辑上属于这个类 --> C[staticmethod]
    Q -- 我要定义契约让别人实现 --> D[ABC + abstractmethod]
```

### 📝 动手练习

1. 写 `Employee` 类：`from_string("张三,28,工程师")` 类方法工厂；`@staticmethod hourly_rate(level)` 返回时薪表；实例方法 `monthly_pay()`。
2. 定义 `PaymentMethod` ABC（`pay(amount)` 抽象方法），实现 `Alipay`、`WeChatPay`、`CreditCard` 三个子类，写函数遍历支付。
3. 思考题：`dict.fromkeys()` 是类方法吗？为什么 `dict` 从字符串、从键值对列表创建有多种方式，而不是多个 `__init__`？

---
**下一章（终章）** 👉 `07_进阶特性.ipynb`：dataclass、组合优于继承、Mixin、`__new__`、描述符与元类，以及 SOLID 设计原则。